In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


In [2]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [3]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER
from utils.res_cluster_picker import pick_max_avg_residual_cluster, pick_max_max_residual_cluster

In [4]:
from ldpc.bp_decoder import BpDecoder

In [5]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [6]:
decoder = BpDecoder(H)

In [7]:
n = 486 # length of message
n_frames = 1000
max_iter = 30

message = np.random.randint(0, 2, (n_frames, n))
print("Message shape:", message.shape)

Message shape: (1000, 486)


In [8]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (1000, 648)


In [9]:
m, _ = H.shape
arr = np.arange(m)
clusters = arr.reshape(6, -1)


In [10]:
snrs = [0, 1, 2, 3, 4, 5, 6, 7]
bers = []


for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]
        # times_cluster = [0, 0, 0, 0, 0, 0]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        
        for iter in range(max_iter):
            residuals = decoder.get_residuals()
            cluster_idx, scheduled_cluster = pick_max_avg_residual_cluster(residuals, clusters)

            # print(f"Scheduled cluster at iteration {iter}: {cluster_idx}")
            # times_cluster[cluster_idx] += 1

            llr = decoder.decode_cluster(scheduled_cluster)

        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        # print("Times each cluster was scheduled:", times_cluster)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")
    


BER at SNR 0 dB: 0.1594485596707819
BER at SNR 1 dB: 0.130559670781893
BER at SNR 2 dB: 0.10202880658436214
BER at SNR 3 dB: 0.07104115226337449
BER at SNR 4 dB: 0.02965843621399177
BER at SNR 5 dB: 0.001866255144032922
BER at SNR 6 dB: 0.0
BER at SNR 7 dB: 0.0


In [11]:
#times_cluster = [0, 0, 0, 0, 0, 0]


for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]
        # times_cluster = [0, 0, 0, 0, 0, 0]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        
        for iter in range(max_iter):
            residuals = decoder.get_residuals()
            cluster_idx, scheduled_cluster = pick_max_avg_residual_cluster(residuals, clusters)

            # print(f"Scheduled cluster at iteration {iter}: {cluster_idx}")
            # times_cluster[cluster_idx] += 1

            llr = decoder.decode_cluster(scheduled_cluster)

        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        # print("Times each cluster was scheduled:", times_cluster)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR 0 dB: 0.15929218106995885
BER at SNR 1 dB: 0.13065843621399176
BER at SNR 2 dB: 0.10189300411522634
BER at SNR 3 dB: 0.07024691358024691
BER at SNR 4 dB: 0.029189300411522633
BER at SNR 5 dB: 0.0018333333333333333
BER at SNR 6 dB: 2.05761316872428e-06
BER at SNR 7 dB: 0.0
